# Elo on the matches table
`elo_sqrt` pre-match ratings (each team's rating just *before* the match, so the result is not leaked) can be written onto `wyscout.matches` as `home_team_elo` and `away_team_elo`. Men and women are rated separately with the tuned settings. Only played regular-season matches get a rating; everything else stays NULL.

`wyscout/load.py` reloads matches with INSERT OR REPLACE, which resets these columns, so run the export again after loading matches.

In [1]:
%matplotlib inline
import sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
# make the repo root importable no matter which folder the notebook is opened from
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "ratings").is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from ratings import base
from ratings.data import load_matches
from ratings.tune import PARAMS_PATH, load_best
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from ratings.export_elo import both_genders_elo, write_to_database
from ratings.data import connect

## 1. Work out the ratings (writes nothing)

In [2]:
elo = both_genders_elo()
print(len(elo), "matches rated")
display(elo.describe().round(1))

34926 matches rated


,match_id,home_team_elo,away_team_elo
count,34926.0,34926.0,34926.0
mean,5508347.0,1549.6,1527.0
std,254961.1,222.2,214.9
min,2667670.0,574.4,614.8
25%,5277017.2,1411.2,1398.0
50%,5509622.5,1533.6,1510.4
75%,5730338.8,1676.1,1645.8
max,5937234.0,2485.7,2481.7


## 2. Write to the database
**This changes the database.** Set `WRITE_TO_DATABASE = True` to run it.

In [3]:
WRITE_TO_DATABASE = False
if WRITE_TO_DATABASE:
    con = connect()
    n_written = write_to_database(con, elo)
    total, with_elo = con.execute("SELECT count(*), count(home_team_elo) FROM wyscout.matches").fetchone()
    print(f"wrote {n_written} rows; wyscout.matches has {with_elo}/{total} rows with Elo")
else:
    print("not written (WRITE_TO_DATABASE = False)")

not written (WRITE_TO_DATABASE = False)
